# <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

In [ ]:
from core import Workflow
from core.gui import PyironFlow

## <font style="font-family:roboto;color:#455e6c"> Calculate free energy with temperature for solid</font> 

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

By now we have seen how the GUI works. We try to calculate the free energy of an Al FCC structure.

Helpful nodes (can be searched for in the Node library):
- `atomistic -> structure -> build -> Bulk` to create a structure. Make sure to check `cubic` box.
- `atomistic -> structure -> transform -> Repeat` to create a supercell. A 3x3x3 cell should work for this example.
- `atomistic -> calculator -> calphy -> InputClass` to set inputs for free energy calculation.
- `atomistic -> calculator -> calphy -> SolidFreeEnergyWithTemp` for calculating the free energy.
- `atomistic -> engine -> lammps -> ListPotentials` to get the list of potentials for LAMMPS, for this specific structure.
- `basic -> list-> PickElement` to pick a potential from the list of potentials. Here, we choose the `2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1` potential for quick calculations.
- `atomistic -> engine -> eam -> EAM` to provide the potential to the calphy calculations.
- `atomistic -> calculator -> calphy -> PlotFreeEnergy` for a free energy plot.

</div>

In [ ]:
wf_free_energy_unary = Workflow("Free_Energy_Unary")
wf_free_energy_unary_solution = Workflow("Free_Energy_Unary_Solution")

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_free_energy_unary = PyironFlow(
    wf_list=[wf_free_energy_unary, wf_free_energy_unary_solution],
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)
pf_free_energy_unary.gui

A preview of the solution:
![thermo](img/Thermo_wf1.png)

## <font style="font-family:roboto;color:#455e6c"> Calculate melting temperature </font> 

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>

Now we take a step forward and calculate the melting temperature. The melting temperature is the temperature at which the free energies of the solid and liquid phases are equal. 

In addition to the nodes above, the following would be useful:
- The steps for solid free energy are same as above, but with good temperature window to be set in the `InputClass` would be 900-1100 K.
- For liquid, we need to create a structure first. There are two ways to do it:
    - either check the `melting_cycle` box in `InputClass`
    - or use `atomistic -> structure -> transform -> Rattle` with `stdev` of approximately 0.6 to create a structure, similar to liquid.
- `atomistic -> calculator -> calphy -> LiquidFreeEnergyWithTemp` for calculating the free energy of the liquid.
- `atomistic -> thermodynamics -> calphy -> LiquidFreeEnergyWithTemp` for calculating the free energy of the liquid phase
- A good temperature window to be set in the `InputClass` would be 800-1100 K.
- `thermodynamics -> landau -> TemperatureDependentLinePhase` to store the free energies and temperatures in an object for future use.
- `thermodynamics -> landau -> TransitionTemperature` to calculate the transition temperature between the solid and the liquid, and to plot the free energies.

</div>

In [ ]:
wf_transition_temp_unary = Workflow("Transition_Temp_Unary")
wf_transition_temp_unary_solution = Workflow("Transition_Temp_Unary_Solution")

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_transition_temp_unary = PyironFlow(
    wf_list=[wf_transition_temp_unary, wf_transition_temp_unary_solution], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_transition_temp_unary.gui

A preview of the solution:
![transition](img/Thermo_wf2.png)

## <font style="font-family:roboto;color:#455e6c"> Calculating a binary phase diagram </font> 

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">

Now that we have seen the thermodynamic functions in Landau, we will make a simple eutectic binary phase diagram with Landau.

- Instead of calculating free energies for a specific structure, we use model phases, whose energies and entropies can be set manually.

- We input the internal energy $U$ and the entropy $S$, and the free energy for a temperature $T$ gets computed using standard the equation below:

$$
F = U - T \cdot S
$$

</div>

In [ ]:
wf_phase_diagram_binary = Workflow("Phase_Diagram_Binary")
wf_phase_diagram_binary_solution = Workflow("Phase_Diagram_Binary_Solution")

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_phase_diagram_binary = PyironFlow(
    wf_list=[wf_phase_diagram_binary, wf_phase_diagram_binary_solution], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_phase_diagram_binary.gui

A preview of the solution:
![phase](img/Thermo_wf3.png)

## <font style="font-family:roboto;color:#455e6c"> MgCa phase diagram </font> 

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">

Now we will take the same approach to calculate a production ready phase diagram with real data, in this case the MgCa system. The data can be found at `data/MgCaFreeEnergies.pckl.gz` ([published data](https://github.com/eisenforschung/mgalca-mtp-data)).

The free energies of the phases have been calculated using Calphy and stored already in a Landau dataframe. The plotting of the phase diagram is once again done using Landau.

</div>

In [ ]:
wf_phase_diagram_MgCa_solution = Workflow("Phase_Diagram_MgCa_Solution")

In [ ]:
workflow_path = "pyiron_core_data/workflows"
pf_phase_diagram_MgCa = PyironFlow(
    wf_list=[wf_phase_diagram_MgCa_solution], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_phase_diagram_MgCa.gui

A preview of the solution:
![phasediagram](img/Thermo_wf4.png)

---
---

### <font style="font-family:roboto;color:#455e6c"> Software used in this notebook </font>  

- [calphy](https://calphy.org)
- [landau](https://github.com/eisenforschung/landau)
- [LAMMPS](https://www.lammps.org/)
- [pyiron](https://pyiron.org/)
- [pyiron_workflow](https://github.com/pyiron/pyiron_workflow)

![roll](img/logo_roll.png)